In [1]:
import torch
import torch.nn as nn
import torchaudio
import torch.nn.functional as F
import torchaudio.transforms as T
from typing import List, Dict, Optional, Iterable, Tuple

import os, re, random
import numpy as np
import sklearn
import itertools
import time

import pickle
from tqdm.auto import tqdm
from IPython.display import clear_output
import IPython.display as ipd
import gc
import matplotlib.pyplot as plt
import wandb

print(torch.__version__)
print(torchaudio.__version__)

import sys
sys.path.append("/home/iblitvinov@vniia.int/projects/Voice-commands-recognition")

torch.cuda.empty_cache()
gc.collect()

2.2.0
2.2.0


0

In [2]:
random.seed(123456)
np.random.seed(123456)
torch.manual_seed(123456)

use_cuda = torch.cuda.is_available()
device = "cuda:3"

def create_plots(signal, feature_map, name_fig, sample_rate=16000, 
                            signal_plot_dir='signal_plots', 
                            spec_plot_dir='spec_plots', 
                            mfcc_plot_dir='mfcc_plots',
                            notebook_path='/home/iblitvinov@vniia.int/projects/Voice-commands-recognition/notebooks'):
    
    signal = signal.cpu().numpy()

    num_frames = signal.size
    time_axis = torch.arange(0, num_frames) / sample_rate
    plt.plot(time_axis, signal, linewidth=1)
    plt.xlabel('Time')
    plt.title('Signal')
    plt.grid()
    plt.savefig(os.path.join(notebook_path, signal_plot_dir, name_fig))
    plt.clf()

    plt.specgram(signal, Fs=sample_rate)
    plt.xlabel('Time')
    plt.title('Spectrogram')
    plt.savefig(os.path.join(notebook_path, spec_plot_dir, name_fig))
    plt.clf()

    plt.imshow(feature_map.cpu(), interpolation='nearest', origin='lower', aspect='auto')
    plt.xlabel('Frame')
    plt.title('MFCC')
    plt.savefig(os.path.join(notebook_path, mfcc_plot_dir, name_fig))
    plt.clf()
    
def plot_waveform(waveform, mfcc, sample_rate=16000, title="Waveform", xlim=None):
    waveform = waveform.cpu().numpy()

    num_frames = waveform.size
    time_axis = torch.arange(0, num_frames) / sample_rate

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))
    ax1.plot(time_axis, waveform, linewidth=1)
    ax1.set_xlabel('Time')
    ax1.set_title('Signal')

    ax2.specgram(waveform, Fs=sample_rate)
    ax2.set_xlabel('Time')
    ax2.set_title('Spectrogram')

    ax3.imshow(mfcc.cpu(), interpolation='nearest', origin='lower', aspect='auto')
    ax3.set_xlabel('Frame')
    ax3.set_title('MFCC')

    plt.suptitle(title)
    plt.show()
    
    ax1.grid(True)
    if xlim:
        ax1.set_xlim(xlim)
        
    fig.suptitle(title)
    plt.show()



### Translit labels to English

In [3]:
data_path = '/home/iblitvinov@vniia.int/projects/stock_data_last_export/'
train_data_list = []
valid_data_list = []
with open(os.path.join(data_path, 'train_data_base_audio.pickle'), 'rb') as fh:
    train_data_list = pickle.load(fh)

with open(os.path.join(data_path, 'valid_data_base_audio.pickle'), 'rb') as fh:
    valid_data_list = pickle.load(fh)

len(train_data_list), len(valid_data_list)

(29721, 544)

In [4]:
# from transliterate import translit

# data_path = '/home/iblitvinov@vniia.int/projects/augmented_data_last_export'
# data_list = []
# new_data_list = []
# with open(os.path.join(data_path, 'data_base_audio.pickle'), 'rb') as fh:
#     data_list = pickle.load(fh)

# print(data_list)

# for example in data_list:
#     curr_sub_path = os.path.join(data_path, "/home/iblitvinov@vniia.int/projects/augmented_data_last_export")
#     new_label = translit(example['label'], reversed=True)
#     new_name = translit(example['name'], reversed=True)

#     try:
#         os.rename(os.path.join(curr_sub_path, example['name']), os.path.join(curr_sub_path, new_name))
#         new_data_list.append({'name': new_name, 'label': new_label})
#     except FileNotFoundError:
#         print("Not opened")
#         pass

# with open(os.path.join(data_path, 'data_base_audio.pickle'), 'wb') as f:
#     pickle.dump(new_data_list, f)

In [5]:
rus_classes = [
 'Домой',
 'Двигаться',
 'Найти',
 'Опустить',
 'Остановиться',
 'Открыть', 
 'Поднять',
 'Сменить',
 'Сохранить',
 'Старт',
 'Стоп',   
 'Влево',
 'Вниз',   
 'Вправо',
 'Вверх',  
 'Загрузить',
 'Захватить',   
 'Закрыть'
 ]

all_labels = set()
for example in train_data_list:
    all_labels.add(example['label'])
token_to_idx = {x: idx for idx, x in enumerate(all_labels)}
all_labels, token_to_idx

({'Domoj',
  "Dvigat'sja",
  'Najti',
  "Opustit'",
  "Ostanovit'sja",
  "Otkryt'",
  "Podnjat'",
  "Smenit'",
  "Sohranit'",
  'Start',
  'Stop',
  'Vlevo',
  'Vniz',
  'Vpravo',
  'Vverh',
  "Zagruzit'",
  "Zahvatit'",
  "Zakryt'"},
 {'Vpravo': 0,
  "Dvigat'sja": 1,
  'Vverh': 2,
  "Otkryt'": 3,
  "Zakryt'": 4,
  "Podnjat'": 5,
  "Ostanovit'sja": 6,
  "Opustit'": 7,
  'Vlevo': 8,
  "Zahvatit'": 9,
  "Sohranit'": 10,
  "Zagruzit'": 11,
  'Domoj': 12,
  'Najti': 13,
  'Vniz': 14,
  'Start': 15,
  "Smenit'": 16,
  'Stop': 17})

In [6]:
# print( len(data_list) )

# for sound in tqdm(data_list):
#     data_subpath = os.path.join(data_path, sound['name'])
#     try:
#         audio = torchaudio.load(data_subpath)
#         if audio[0].size(1) == 0:
#             print(audio[0].size(1))
#             data_list.remove(sound)
#             os.remove(data_subpath)
#             print(f'Removed {sound["name"]}')
#     except RuntimeError:
#         print(sound)
#         os.remove(data_subpath)
#         data_list.remove(sound)

# with open(os.path.join(data_path, 'data_base_audio.pickle'), 'wb') as f:
#     pickle.dump(data_list, f)
    
# print( len(data_list) )

In [7]:
# lengths_subsets = {'train': int(0.8 * len(data_list)), 'valid': round(0.1 * len(data_list)), 'test': round(0.1 * len(data_list))}
# train_vaild_subset, test_subset = torch.utils.data.random_split(data_list, 
#                                                                 [lengths_subsets['train']+lengths_subsets['valid'], lengths_subsets['test']])
# train_subset, valid_subset = torch.utils.data.random_split(train_vaild_subset, 
#                                                                 [lengths_subsets['train'], lengths_subsets['valid']])
# lengths_subsets

In [8]:
import gc

class Sound_dataset_commands(torch.utils.data.Dataset):
    def __init__(self, rootdir, subset, transform=None):
        self.transform = transform
        self.rootdir = rootdir
        self.subset = subset
        self.n_subset = len(self.subset)
        self.token_to_idx = {x: idx for idx, x in enumerate(all_labels)}
        self.idx_to_token = {idx: x for idx, x in enumerate(all_labels)}

        self.preprocessed_dataset = self._preprocessing()
        torch.cuda.empty_cache()
        gc.collect()
        self.preprocessed_dataset = self.preprocessed_dataset

    def _preprocessing(self):
        dataset = []
        for sound in tqdm(self.subset):
            name, label = sound.values()
            signal, _ = torchaudio.load(os.path.join(self.rootdir, name))
            signal = signal[0]
            if self.transform:
                feature_map = self.transform(signal)
#               signal = self.transform.transforms[0](signal)
            idx_label = self.token_to_idx[label]
            dataset.append((feature_map.to(device), idx_label, signal.to(device)))
        
        return dataset

    def __getitem__(self, index):
        feature_map, idx_label, signal = self.preprocessed_dataset[index]
        name, label = self.subset[index].values()

        return feature_map, idx_label, signal, name, label

    def __len__(self):
        return len(self.subset)

In [9]:
from models.model import Speech_recognition_model

def create_data_sets(n_mfcc):
    n_fft = 480
    win_length = None
    hop_length = 160
    mfcc_transform = T.MFCC(
        sample_rate=16000,
        n_mfcc=n_mfcc,
        melkwargs={
            "n_fft": n_fft,
            "n_mels": n_mfcc * 2,
            "hop_length": hop_length,
            "f_min": 20,
            "f_max": 4000
        },
    )

    transform = mfcc_transform
    data_set = {
                'train': Sound_dataset_commands(rootdir=data_path, subset=train_data_list, transform=transform),
                'valid': Sound_dataset_commands(rootdir=data_path, subset=valid_data_list, transform=transform),
                # 'test': Sound_dataset_commands(rootdir=data_path, subset=test_subset, transform=transform)
            }
    return data_set

def create_data_loaders(batch_size, data_set):

    def collate_fn(batch):
        X = torch.nn.utils.rnn.pad_sequence([sample[0].transpose(0, 1) for sample in batch], batch_first=True, padding_value=0).unsqueeze(1)
        y = torch.Tensor([sample[1] for sample in batch]).to(torch.long).to(device)

        signal = [sample[2] for sample in batch]
        name = [sample[3] for sample in batch]
        label = [sample[4] for sample in batch]

        return X, y, signal, name, label

    loaders = {
            'train': torch.utils.data.DataLoader(data_set['train'], batch_size=batch_size, shuffle=True, collate_fn=collate_fn),
            'valid': torch.utils.data.DataLoader(data_set['valid'], batch_size=batch_size, shuffle=True, collate_fn=collate_fn),
            # 'test': torch.utils.data.DataLoader(data_set['test'], batch_size=1, shuffle=False, collate_fn=collate_fn)
            }
    return loaders


def create_model_utils(params, weight_decay, amsgrad, max_lr, div_factor, epochs, steps_per_epoch, tl=False, tl_type='freeze', tl_low_lr=1e-8, load_path=None):
    tr_loss = []
    cv_loss = []
    tr_acc = []
    cv_acc = []
    epoch = 0
    model = Speech_recognition_model(**params).to(device)
    criterion = torch.nn.CrossEntropyLoss(reduction='mean').to(device)

    optimizer = torch.optim.Adam(model.parameters(),
                                lr=max_lr/div_factor,
                                weight_decay=weight_decay,
                                amsgrad=amsgrad)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=max_lr,
                            steps_per_epoch=steps_per_epoch,
                            epochs=epochs, div_factor=div_factor)

    if tl:
        model.load_state_dict(torch.load('../../checkpoints/sub_model_asr_2.pth', map_location=device), strict=False)
        if tl_type == 'freeze':
            for param in list(model.parameters())[:28]: #28
                param.requires_grad = False

        elif tl_type == 'low_step':
            optimizer = torch.optim.Adam(params=[
                                            {"params": list(model.parameters())[28:]},
                                            {"params": list(model.parameters())[:28], "lr": tl_low_lr}
                                        ], lr=max_lr/div_factor, weight_decay=weight_decay, amsgrad=amsgrad)

        elif tl_type == 'non_freeze':
            pass

    if load_path is not None:
        checkpoint = torch.load(load_path)

        model.load_state_dict(checkpoint['state_dict'], map_location=device)
        optimizer.load_state_dict(checkpoint['optim_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_dict'])
        tr_loss = checkpoint['tr_loss']
        cv_loss = checkpoint['cv_loss']
        epoch = checkpoint['epoch']
        tr_acc = checkpoint['tr_accuracy']
        cv_acc = checkpoint['cv_accuracy']
    
    return model, criterion, optimizer, scheduler, tr_loss, cv_loss, epoch, tr_acc, cv_acc

In [10]:
from models.model import Speech_recognition_model
import yaml

# Сделать разные конфиги
params = None
with open("../configs/model_params_res.yaml", "r") as stream:
    try:
        params = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        print(exc)

In [11]:
params

{'Architecture': {'CNN_params': {'in_channels': 1,
   'out_channels': 64,
   'kernel_size': 3,
   'stride': 2,
   'padding': 1},
  'ResCNN_params': {'in_channels': 64,
   'out_channels': 64,
   'kernel_size': 3,
   'stride': 1,
   'dropout': 0.255,
   'n_feats': 16,
   'padding': 1,
   'n_cnn_layers': 4},
  'Fully_connected_params': {'in_features': 1024, 'out_features': 512},
  'RNN_params': {'input_size': 512,
   'hidden_size': 512,
   'num_layers': 2,
   'bidirectional': True,
   'dropout': 0.27,
   'rnn_type': 'lstm'},
  'Attention_params': {'feature_dim': 512, 'step_dim': 4},
  'Classifier_params': {'in_features': 512,
   'out_features': 256,
   'dropout': 0.4,
   'n_class': 18}},
 'Settings': {'Optimizer_Scheduler': {'lr_rate': '5e-4',
   'weight_decay': 0.0005,
   'amsgrad': False,
   'max_lr': 0.07,
   'div_factor': 125,
   'max_norm': 1.4},
  'Other': {'epochs': 100, 'batch_size': 64}}}

In [12]:
!wandb login --relogin a4fc0db907801a21173615cb84708e1774b986a4

wandb: Appending key for api.wandb.ai to your netrc file: /home/iblitvinov@vniia.int/.netrc


In [18]:
sweep_configuration = {
    'method': 'bayes',
        'metric': {
        'goal': 'minimize', 
        'name': 'valid_loss'
        },
    'parameters': 
        {  "n_mfcc": {"values": [32]},
           "CNN_out_channels": {'values': [32, 64]}, "CNN_kernel_size": {'values': [3, 5]},
            "ResCNN_kernel_size": {'values': [3, 5]}, "ResCNN_dropout": {'min': 0.1, 'max': 0.6}, "ResCNN_n_cnn_layers": {'min': 1, "max": 5},
            "FC_out_features": {"values": [128, 256, 512, 1024]},
            "RNN_num_layers": {'min': 2, 'max': 6}, "RNN_bi": {'values': [True]}, "RNN_dropout": {'min': 0.1, 'max': 0.5},
            "Classifier_dropout": {'min': 0.25, 'max': 0.75},
            'weight_decay': {'min': 0.0001, 'max': 0.0007}, 'amsgrad': {'values': [False, True]}, 'max_lr': {'min': 0.01, 'max': 0.1}, 'div_factor': {'min': 100, 'max': 300}, 'max_norm': {'min': 0.7, 'max': 1.5}, 
            'epochs': {'values': [35]}, 'batch_size': {'values': [64, 128, 256]}
        }
}

sweep_id = wandb.sweep(sweep=sweep_configuration, project="Commands Recognition")

Create sweep with ID: u4v09ycy
Sweep URL: https://wandb.ai/litvan/Commands%20Recognition/sweeps/u4v09ycy


In [19]:
def get_accuracy(y_pred, y_test):
    correct_results_sum = (y_pred == y_test).sum().float()
    acc = correct_results_sum/(y_test.size(0))
    
    return acc

In [15]:
data_set = create_data_sets(32)

  0%|          | 0/29721 [00:00<?, ?it/s]

  0%|          | 0/544 [00:00<?, ?it/s]

In [20]:
signal_plot_dir='signal_plots'
spec_plot_dir='spec_plots'
mfcc_plot_dir='mfcc_plots'
notebook_path = '/home/iblitvinov@vniia.int/projects/Voice-commands-recognition/notebooks'


def sweep_func():
    torch.cuda.empty_cache()
    gc.collect()
    
    wandb.init(project="Commands Recognition")
    columns = ["name", "signal_plot", "spec_plot", "mfcc_plot", "pred_label", "true_label"]
    valid_table = wandb.Table(columns=columns)
    loaders = create_data_loaders(wandb.config['batch_size'], data_set)
    epochs = wandb.config['epochs']
    max_norm = wandb.config['max_norm']
    params['Architecture']['CNN_params']['kernel_size'] = wandb.config['CNN_kernel_size']
    params['Architecture']['CNN_params']['padding'] = wandb.config['CNN_kernel_size'] // 2
    params['Architecture']['CNN_params']['out_channels'] = wandb.config['CNN_out_channels']
    params['Architecture']['ResCNN_params']['in_channels'] = wandb.config['CNN_out_channels']
    params['Architecture']['ResCNN_params']['out_channels'] = wandb.config['CNN_out_channels']
    params['Architecture']['ResCNN_params']['kernel_size'] = wandb.config['ResCNN_kernel_size']
    params['Architecture']['ResCNN_params']['padding'] = wandb.config['ResCNN_kernel_size'] // 2
    params['Architecture']['ResCNN_params']['dropout'] = wandb.config['ResCNN_dropout'] 
    params['Architecture']['ResCNN_params']['n_cnn_layers'] = wandb.config['ResCNN_n_cnn_layers']
    params['Architecture']['ResCNN_params']['n_feats'] = wandb.config['n_mfcc'] // 2
    params['Architecture']['Fully_connected_params']['in_features'] = wandb.config['CNN_out_channels'] * params['Architecture']['ResCNN_params']['n_feats']
    params['Architecture']['Fully_connected_params']['out_features'] = wandb.config['FC_out_features']
    params['Architecture']['RNN_params']['input_size'] = wandb.config['FC_out_features']
    params['Architecture']['RNN_params']['hidden_size'] = wandb.config['FC_out_features']
    params['Architecture']['RNN_params']['num_layers'] = wandb.config['RNN_num_layers']
    params['Architecture']['RNN_params']['bidirectional'] = wandb.config['RNN_bi']
    params['Architecture']['RNN_params']['dropout'] = wandb.config['RNN_dropout']
    params['Architecture']['Attention_params']['feature_dim'] = wandb.config['FC_out_features']
    params['Architecture']['Attention_params']['step_dim'] = params['Architecture']['RNN_params']['num_layers'] * 2
    params['Architecture']['Classifier_params']['in_features'] = wandb.config['FC_out_features']
    params['Architecture']['Classifier_params']['dropout'] = wandb.config['Classifier_dropout']
    params['Architecture']['Classifier_params']['out_features'] = params['Architecture']['Classifier_params']['in_features'] // 2
    params['Settings']['Other']['batch_size'] = wandb.config['batch_size']


    model, criterion, optimizer, scheduler, tr_loss, cv_loss, epoch, tr_acc, cv_acc = create_model_utils(params['Architecture'],
                                                                wandb.config['weight_decay'],
                                                                wandb.config['amsgrad'],
                                                                wandb.config['max_lr'],
                                                                wandb.config['div_factor'],
                                                                epochs,
                                                                len(loaders['train']))
    
    def get_nn_params_stat(model: nn.Module) -> None:
        def iter_mul(inputs: Iterable) -> int:
            mul = 1
            for elem in inputs:
                mul *= elem
            return mul
        
        shapes = [p.shape for p in model.parameters()]
        for p_shape in shapes:
            print(p_shape)
        total_count = sum([iter_mul(p_shape) for p_shape in shapes])
        print('Total params:', total_count)
        print('Model param size in Mb:', total_count * 4 / (2 ** 20))

    print( params )
    print( get_nn_params_stat(model) )
    
    def run_one_epoch(epoch, cross_valid=False, print_freq=8):
        start = time.time()
        total_loss = 0
        total_accuracy = 0
        total_n = 0

        data_loader = loaders['train'] if not cross_valid else loaders['valid']

        true_labels = []
        pred_labels = []
        
        for data in tqdm(data_loader):
            feature_map, idx_label, signal, name, label = data

            outputs = model(feature_map)
            loss = criterion(outputs, idx_label)

            total_n += 1

            if not cross_valid:
                optimizer.zero_grad()
                loss.backward()
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(),
                                                            max_norm)
                optimizer.step()

            total_loss += loss.item()
            preds = torch.argmax(F.softmax(outputs, dim=1), dim=1)
            accuracy = get_accuracy(preds, idx_label)
            total_accuracy += accuracy
            optim_state = optimizer.state_dict()
            curr_lr = optim_state['param_groups'][0]['lr']

            if total_n % print_freq == 0:
                    print('Epoch {0} | Iter {1} | Average Loss {2:.3f} | '
                        'Current Loss {3:.6f} | Current accuracy {4:.6f} | Current lr {5:.5f} | {6:.1f} ms/batch'.format(
                            epoch + 1, total_n + 1, total_loss / (total_n + 1),
                            loss.item(), accuracy, curr_lr, 1000 * (time.time() - start) / (total_n + 1)),
                        flush=True)

            if cross_valid:

                name_fig = f'{name[0].replace("/", "_").split(".")[0]}.png'
                create_plots(signal[0], feature_map[0][0], name_fig, notebook_path=notebook_path)    
                temp = [name[0], 
                    # wandb.Audio(signal[0].cpu(), sample_rate=16000), 
                    wandb.Image(os.path.join( notebook_path, signal_plot_dir, name_fig )),
                    wandb.Image(os.path.join( notebook_path, spec_plot_dir, name_fig )),
                    wandb.Image(os.path.join( notebook_path, mfcc_plot_dir, name_fig )),
                    preds[0],
                    idx_label[0]]
                valid_table.add_row(*temp)

                true_labels.extend(idx_label.tolist())
                pred_labels.extend(preds.tolist())

        if cross_valid:
            wandb.log({"conf_mat" : wandb.plot.confusion_matrix(probs=None,
                            y_true=true_labels, preds=pred_labels,
                            class_names=rus_classes)})

        return total_loss / total_n, total_accuracy / total_n
    
    checkpoint = False
    visdom = True
    save_folder = '../checkpoints/'
    best_val_loss = 10

    wandb.watch(model, log_freq=1, log='all')
    for epoch in tqdm(np.arange(epoch, epochs)):
        print("Training...")
        model.train()
        start = time.time()
        tr_avg_loss, tr_avg_acc = run_one_epoch(epoch)
        tr_loss.append(tr_avg_loss)
        tr_acc.append(tr_avg_acc)

        print('-' * 85)
        print('Train Summary | End of Epoch {0} | Time {1:.2f}s | '
                    'Train Loss {2:.3f} | Accuracy {3:.3f}'.format(
                        epoch + 1, time.time() - start, tr_avg_loss, tr_avg_acc))
        print('-' * 85)

        if checkpoint:
            file_path = os.path.join(
            save_folder, 'epoch_{0}_loss_{1:.4f}.pth.tar'.format(epoch + 1, cv_avg_loss))
            torch.save(model.serialize(model, optimizer, scheduler, epoch + 1,
                                            tr_loss=tr_loss,
                                            cv_loss=cv_loss),
                    file_path)
            print('Saving checkpoint model to %s' % file_path)

        print('Cross validation...')
        model.eval()  # Turn off Batchnorm & Dropout
        cv_avg_loss, cv_avg_acc = run_one_epoch(epoch, cross_valid=True)
        cv_loss.append(cv_avg_loss)
        cv_acc.append(cv_avg_acc)
        
        print('-' * 185)
        print('Valid Summary | End of Epoch {0} | Time {1:.2f}s | '
                'Valid Loss {2:.3f} | Accuracy {3:.3f}'.format(
                    epoch + 1, time.time() - start, cv_avg_loss, cv_avg_acc))
        print('-' * 185)

        # Save the best model
        if checkpoint and cv_avg_loss < best_val_loss:
            best_val_loss = cv_avg_loss
            model_path = 'epoch_{0}_loss_{1:.4f}_best.pth.tar'.format(epoch + 1, cv_avg_loss)
            file_path = os.path.join(save_folder, model_path)
            torch.save(model.serialize(model, optimizer, scheduler, epoch + 1,
                                        tr_losses=tr_loss,
                                        cv_losses=cv_loss,
                                        tr_accuracy=tr_acc,
                                        cv_accuracy=cv_acc),
                    file_path)
            print("Find better validated model, saving to %s" % file_path)

        wandb.log({"epoch": epoch})
        wandb.log({"train_loss": tr_avg_loss})
        wandb.log({"valid_loss": cv_avg_loss})
        wandb.log({"train_accuracy": tr_avg_acc})
        wandb.log({"valid_accuracy": cv_avg_acc})
        wandb.run.log({"valid_inference" : valid_table}) 

In [ ]:
wandb.agent(sweep_id, function=sweep_func, count=1000)

wandb: Agent Starting Run: 48c8u7ud with config:
wandb: 	CNN_kernel_size: 3
wandb: 	CNN_out_channels: 32
wandb: 	Classifier_dropout: 0.7447130838448621
wandb: 	FC_out_features: 128
wandb: 	RNN_bi: True
wandb: 	RNN_dropout: 0.33860586309031027
wandb: 	RNN_num_layers: 6
wandb: 	ResCNN_dropout: 0.3000679360069157
wandb: 	ResCNN_kernel_size: 5
wandb: 	ResCNN_n_cnn_layers: 2
wandb: 	amsgrad: False
wandb: 	batch_size: 128
wandb: 	div_factor: 276
wandb: 	epochs: 35
wandb: 	max_lr: 0.09839665383818452
wandb: 	max_norm: 1.2125765387865823
wandb: 	n_mfcc: 32
wandb: 	weight_decay: 0.00050492411262062
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: litvan. Use `wandb login --relogin` to force relogin
wandb: WARNING Ignored wandb.init() arg project when running a sweep.


Not setting weights for type <class 'torch.nn.modules.dropout.Dropout'>
Not setting weights for type <class 'torch.nn.modules.dropout.Dropout'>
Not setting weights for type <class 'torch.nn.modules.normalization.LayerNorm'>
Not setting weights for type <class 'models.cnn.CNNLayerNorm'>
Not setting weights for type <class 'torch.nn.modules.normalization.LayerNorm'>
Not setting weights for type <class 'models.cnn.CNNLayerNorm'>
Not setting weights for type <class 'models.cnn.ResidualCNN'>
Not setting weights for type <class 'torch.nn.modules.dropout.Dropout'>
Not setting weights for type <class 'torch.nn.modules.dropout.Dropout'>
Not setting weights for type <class 'torch.nn.modules.normalization.LayerNorm'>
Not setting weights for type <class 'models.cnn.CNNLayerNorm'>
Not setting weights for type <class 'torch.nn.modules.normalization.LayerNorm'>
Not setting weights for type <class 'models.cnn.CNNLayerNorm'>
Not setting weights for type <class 'models.cnn.ResidualCNN'>
Not setting weig

  0%|          | 0/35 [00:00<?, ?it/s]

Training...


  0%|          | 0/233 [00:00<?, ?it/s]

...

Epoch 23 | Iter 9 | Average Loss 0.362 | Current Loss 0.498359 | Current accuracy 0.882812 | Current lr 0.00036 | 173.9 ms/batch
Epoch 23 | Iter 17 | Average Loss 0.362 | Current Loss 0.459723 | Current accuracy 0.882812 | Current lr 0.00036 | 183.2 ms/batch
Epoch 23 | Iter 25 | Average Loss 0.362 | Current Loss 0.286743 | Current accuracy 0.945312 | Current lr 0.00036 | 186.1 ms/batch
Epoch 23 | Iter 33 | Average Loss 0.360 | Current Loss 0.475942 | Current accuracy 0.875000 | Current lr 0.00036 | 187.1 ms/batch
Epoch 23 | Iter 41 | Average Loss 0.360 | Current Loss 0.353152 | Current accuracy 0.945312 | Current lr 0.00036 | 190.0 ms/batch
Epoch 23 | Iter 49 | Average Loss 0.358 | Current Loss 0.373785 | Current accuracy 0.906250 | Current lr 0.00036 | 189.8 ms/batch
Epoch 23 | Iter 57 | Average Loss 0.361 | Current Loss 0.266064 | Current accuracy 0.937500 | Current lr 0.00036 | 190.9 ms/batch
Epoch 23 | Iter 65 | Average Loss 0.357 | Current Loss 0.400275 | Current accuracy 0.921875

  0%|          | 0/5 [00:00<?, ?it/s]

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Valid Summary | End of Epoch 23 | Time 49.90s | Valid Loss 0.669 | Accuracy 0.870
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Training...


  0%|          | 0/233 [00:00<?, ?it/s]

Epoch 24 | Iter 9 | Average Loss 0.305 | Current Loss 0.301324 | Current accuracy 0.921875 | Current lr 0.00036 | 183.0 ms/batch
Epoch 24 | Iter 17 | Average Loss 0.298 | Current Loss 0.573980 | Current accuracy 0.859375 | Current lr 0.00036 | 189.4 ms/batch
Epoch 24 | Iter 25 | Average Loss 0.340 | Current Loss 0.318146 | Current accuracy 0.945312 | Current lr 0.00036 | 193.6 ms/batch
Epoch 24 | Iter 33 | Average Loss 0.331 | Current Loss 0.261568 | Current accuracy 0.937500 | Current lr 0.00036 | 192.6 ms/batch
Epoch 24 | Iter 41 | Average Loss 0.330 | Current Loss 0.283388 | Current accuracy 0.929688 | Current lr 0.00036 | 195.0 ms/batch
Epoch 24 | Iter 49 | Average Loss 0.340 | Current Loss 0.481555 | Current accuracy 0.890625 | Current lr 0.00036 | 193.7 ms/batch
Epoch 24 | Iter 57 | Average Loss 0.349 | Current Loss 0.399335 | Current accuracy 0.953125 | Current lr 0.00036 | 194.3 ms/batch
Epoch 24 | Iter 65 | Average Loss 0.354 | Current Loss 0.386831 | Current accuracy 0.898438